In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

# Input Data
time = np.array(Friction_Data)
cof = np.array(Friction_Data_Rolling_Average)

# Convert Quantities
displacement = time * 0.1 # Convert time to displacement (15 mm in 150 s)
shear_stress = cof * (235 / 78.54) #Convert COF to shear stress (MPa)

## Detect Peaks and Troughs
peaks, _ = find_peaks(shear_stress,prominence=0.01) 
troughs, _ = find_peaks(-shear_stress, prominence=0.01) 

# Identify Stick-Slip Events
results = []
peak_points = []
trough_points = []
window = 0.2  # mm

for peak in peaks:

    
    nearby_troughs = troughs[
        (troughs > peak) &
        (displacement[troughs] - displacement[peak] <= window)
    ]

    if len(nearby_troughs) == 0:
        continue

    trough = nearby_troughs[np.argmin(shear_stress[nearby_troughs])]

    drop = shear_stress[peak] - shear_stress[trough]

    if drop > 0.06:

        peak_points.append(peak)
        trough_points.append(trough)

        results.append([
            displacement[peak],
            displacement[trough],
            displacement[trough] - displacement[peak],
            drop])

df = pd.DataFrame(
    results,
    columns=[
        "Peak displacement (mm)",
        "Trough displacement (mm)",
        "Displacement difference (mm)",
        "Stress drop (MPa)"])

print(df.head())
print(f"Found {len(results)} stick-slip events")
